<a href="https://colab.research.google.com/github/falcon-140/CSCI226-A2-/blob/main/CSCI226(Dgroup).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
!pip -q install psycopg2-binary SQLAlchemy pandas

In [118]:
import psycopg2, pandas as pd

conn = psycopg2.connect(
    dbname="neondb",
    user="neondb_owner",
    password="npg_bPx2ApvUuE1D",
    host="ep-late-band-adqwtbg7-pooler.c-2.us-east-1.aws.neon.tech",
    port="5432"
)

cur = conn.cursor()


In [32]:
cur.execute("""
    CREATE TABLE IF NOT EXISTS Classes (
        class VARCHAR(20) PRIMARY KEY,
        type VARCHAR(5),
        country VARCHAR(20),
        numGuns INT,
        bore INT,
        displacement INT
    );
""")

In [34]:
conn.commit()
print("Table Classes created successfully!")

Table Classes created successfully!


In [99]:
cur.execute("""
    CREATE TABLE IF NOT EXISTS Ships (
        name VARCHAR(30) PRIMARY KEY,
        class VARCHAR(20) ,
        launched INT
    );
""")
conn.commit()
print("Table Ships created successfully!")

Table Ships created successfully!


In [37]:
cur.execute("""
    CREATE TABLE IF NOT EXISTS Battles (
        name VARCHAR(30) PRIMARY KEY,
        date VARCHAR(20)
    );
""")
conn.commit()
print("Table Battles created successfully!")

Table Battles created successfully!


In [90]:
cur.execute("""
    CREATE TABLE Outcomes (
        ship VARCHAR(30),
        battle VARCHAR(30),
        result VARCHAR(20)
    );
""")
conn.commit()
print("Table Outcomes created successfully!")

Table Outcomes created successfully!


In [39]:
cur.execute("""
    INSERT INTO Classes VALUES
        ('Bismarck', 'bb', 'Germany', 8, 15, 42000),
        ('Iowa', 'bb', 'USA', 9, 16, 46000),
        ('Kongo', 'bc', 'Japan', 8, 14, 32000),
        ('North Carolina', 'bb', 'USA', 9, 16, 37000),
        ('Renown', 'bc', 'Gt. Britain', 6, 15, 32000),
        ('Revenge', 'bb', 'Gt. Britain', 8, 15, 29000),
        ('Tennessee', 'bb', 'USA', 12, 14, 32000),
        ('Yamato', 'bb', 'Japan', 9, 18, 65000);
""")
conn.commit()
print("Data inserted into Classes!")


Data inserted into Classes!


In [44]:
cur.execute("""
    INSERT INTO Battles VALUES
        ('Denmark Strait', '5/24–27/41'),
        ('Guadalcanal', '11/15/42'),
        ('North Cape', '12/26/43'),
        ('Surigao Strait', '10/25/44')
    ON CONFLICT (name) DO NOTHING;
""")
conn.commit()
print("Data inserted into Battles!")


Data inserted into Battles!


In [84]:
conn.rollback()

In [104]:
cur.execute("""
    INSERT INTO Outcomes VALUES
        ('Arizona', 'Pearl Harbor', 'sunk'),
        ('Bismarck', 'Denmark Strait', 'sunk'),
        ('California', 'Surigao Strait', 'ok'),
        ('Duke of York', 'North Cape', 'ok'),
        ('Fuso', 'Surigao Strait', 'sunk'),
        ('Hood', 'Denmark Strait', 'sunk'),
        ('King George V', 'Denmark Strait', 'ok'),
        ('Kirishima', 'Guadalcanal', 'sunk'),
        ('Prince of Wales', 'Denmark Strait', 'damaged'),
        ('Rodney', 'Denmark Strait', 'ok'),
        ('Scharnhorst', 'North Cape', 'sunk'),
        ('South Dakota', 'Guadalcanal', 'damaged'),
        ('Tennessee', 'Surigao Strait', 'ok'),
        ('Washington', 'Guadalcanal', 'ok'),
        ('West Virginia', 'Surigao Strait', 'ok'),
        ('Yamashiro', 'Surigao Strait', 'sunk');
""")
conn.commit()
print("Data inserted into Outcomes!")


Data inserted into Outcomes!


In [100]:
cur.execute("""
    INSERT INTO Ships VALUES
        ('California', 'Tennessee', 1921),
        ('Haruna', 'Kongo', 1915),
        ('Hiei', 'Kongo', 1914),
        ('Iowa', 'Iowa', 1943),
        ('Kirishima', 'Kongo', 1915),
        ('Kongo', 'Kongo', 1913),
        ('Missouri', 'Iowa', 1944),
        ('Musashi', 'Yamato', 1942),
        ('New Jersey', 'Iowa', 1943),
        ('North Carolina', 'North Carolina', 1941),
        ('Ramillies', 'Revenge', 1917),
        ('Renown', 'Renown', 1916),
        ('Repulse', 'Renown', 1916),
        ('Resolution', 'Revenge', 1916),
        ('Revenge', 'Revenge', 1916),
        ('Royal Oak', 'Revenge', 1916),
        ('Royal Sovereign', 'Revenge', 1916),
        ('Tennessee', 'Tennessee', 1920),
        ('Washington', 'North Carolina', 1941),
        ('Wisconsin', 'Iowa', 1944),
        ('Yamato', 'Yamato', 1941)
    ON CONFLICT (name) DO NOTHING;
""")
conn.commit()
print("Data inserted into Ships!")


Data inserted into Ships!


In [109]:
conn.rollback()

In [72]:
cur.execute("""
    TRUNCATE TABLE Ships RESTART IDENTITY CASCADE;
""")
conn.commit()
print("Truncated the ship table")

Truncated the ship table


In [81]:
cur.execute("""
    TRUNCATE TABLE Outcomes RESTART IDENTITY CASCADE;
""")
conn.commit()
print("Truncated the outcome table")

Truncated the outcome table


In [89]:
cur.execute("""DROP TABLE IF EXISTS Outcomes;""")

# Query 1

In [119]:
# Query 1: Naval Timeline
print("=== Query 1: Naval Development Timeline ===")
cur.execute("""
    SELECT c.country,
           MIN(s.launched) as oldest_ship_year,
           MAX(s.launched) as newest_ship_year,
           MAX(s.launched) - MIN(s.launched) as years_span
    FROM Ships s
    JOIN Classes c ON s.class = c.class
    GROUP BY c.country
    ORDER BY years_span DESC;
""")
results1 = cur.fetchall()
for row in results1:
    print(f"{row[0]}: {row[1]}-{row[2]} ({row[3]} year span)")


=== Query 1: Naval Development Timeline ===
Japan: 1913-1942 (29 year span)
USA: 1920-1944 (24 year span)
Gt. Britain: 1916-1917 (1 year span)


# Query 2

In [120]:
# Query 2: Sister Ships
print("\n=== Query 2: Sister Ships (Same Class, Close Launch Dates) ===")
cur.execute("""
    SELECT s1.name, s2.name, s1.class, s1.launched, s2.launched
    FROM Ships s1
    JOIN Ships s2 ON s1.class = s2.class
    WHERE s1.name < s2.name
      AND ABS(s1.launched - s2.launched) <= 2
    ORDER BY s1.class;
""")
results2 = cur.fetchall()
for row in results2:
    print(f"{row[0]} & {row[1]} ({row[2]} class): {row[3]} & {row[4]}")


=== Query 2: Sister Ships (Same Class, Close Launch Dates) ===
New Jersey & Wisconsin (Iowa class): 1943 & 1944
Iowa & Wisconsin (Iowa class): 1943 & 1944
Iowa & New Jersey (Iowa class): 1943 & 1943
Iowa & Missouri (Iowa class): 1943 & 1944
Missouri & Wisconsin (Iowa class): 1944 & 1944
Missouri & New Jersey (Iowa class): 1944 & 1943
Hiei & Kirishima (Kongo class): 1914 & 1915
Hiei & Kongo (Kongo class): 1914 & 1913
Kirishima & Kongo (Kongo class): 1915 & 1913
Haruna & Kongo (Kongo class): 1915 & 1913
Haruna & Kirishima (Kongo class): 1915 & 1915
Haruna & Hiei (Kongo class): 1915 & 1914
North Carolina & Washington (North Carolina class): 1941 & 1941
Renown & Repulse (Renown class): 1916 & 1916
Royal Oak & Royal Sovereign (Revenge class): 1916 & 1916
Ramillies & Royal Sovereign (Revenge class): 1917 & 1916
Ramillies & Royal Oak (Revenge class): 1917 & 1916
Ramillies & Revenge (Revenge class): 1917 & 1916
Ramillies & Resolution (Revenge class): 1917 & 1916
Revenge & Royal Sovereign (Rev

# Query 3

In [121]:
# Query 3: Naval Power Index
print("\n=== Query 3: Naval Power Rankings ===")
cur.execute("""
    SELECT c.country,
           COUNT(s.name) as total_ships,
           SUM(c.numGuns) as total_guns,
           SUM(c.displacement) as total_tonnage
    FROM Ships s
    JOIN Classes c ON s.class = c.class
    GROUP BY c.country
    ORDER BY total_tonnage DESC;
""")
results3 = cur.fetchall()
for row in results3:
    print(f"{row[0]}: {row[1]} ships, {row[2]} guns, {row[3]:,} tons")


=== Query 3: Naval Power Rankings ===
USA: 8 ships, 78 guns, 322,000 tons
Japan: 6 ships, 50 guns, 258,000 tons
Gt. Britain: 7 ships, 52 guns, 209,000 tons


# exercise 6.2.3

In [106]:
cur.execute("""
    SELECT DISTINCT o1.ship
        FROM Outcomes o1, Outcomes o2, Battles b1, Battles b2
        WHERE o1.ship = o2.ship
          AND o1.battle = b1.name
          AND o2.battle = b2.name
          AND o1.result = 'damaged'
          AND b1.date < b2.date;
""")
conn.commit()
results = cur.fetchall()
print("Find those ships that were damaged in one battle, but later fought in another battle.")
print(results)


Find those ships that were damaged in one battle, but later fought in another battle.
[]


In [111]:
cur.execute("""
    SELECT c1.country
          FROM Classes c1
          WHERE c1.type = 'bb'
            AND c1.country IN (
              SELECT c2.country
              FROM Classes c2
              WHERE c2.type = 'bc'
            );
""")
conn.commit()
results = cur.fetchall()
print("Find those countries that have both battleships and battlecruisers.")
print(results)


Find those countries that have both battleships and battlecruisers.
[('Gt. Britain',), ('Japan',)]


In [102]:
cur.execute("""
    SELECT s.name
          FROM Ships s
          JOIN Classes c ON s.class = c.class
          WHERE c.displacement > 35000;
""")
conn.commit()
results = cur.fetchall()
print("Find the ships heavier than 35,000 tons.")
print(results)


Find the ships heavier than 35,000 tons.
[('Iowa',), ('Missouri',), ('Musashi',), ('New Jersey',), ('North Carolina',), ('Washington',), ('Wisconsin',), ('Yamato',)]


# exercise 6.1.4

In [113]:
cur.execute("""
    SELECT class, country
FROM Classes
WHERE numGuns >= 10;
""")
conn.commit()
results = cur.fetchall()
print("Find the class name and country for all classes with at least 10 guns.")
print(results)


Find the class name and country for all classes with at least 10 guns.
[('Tennessee', 'USA')]


In [114]:
cur.execute("""
    SELECT name
FROM Ships
WHERE name LIKE 'R%';
""")
conn.commit()
results = cur.fetchall()
print("Find the names of all ships that begin with the letter R.")
print(results)


Find the names of all ships that begin with the letter R.
[('Ramillies',), ('Renown',), ('Repulse',), ('Resolution',), ('Revenge',), ('Royal Oak',), ('Royal Sovereign',)]


In [116]:
cur.execute("""
    SELECT name
FROM Ships
WHERE LENGTH(name)>= 2;
""")
conn.commit()
results = cur.fetchall()
print("Find the names of all ships whose name consists of three or more words.")
print(results)


Find the names of all ships whose name consists of three or more words.
[('California',), ('Haruna',), ('Hiei',), ('Iowa',), ('Kirishima',), ('Kongo',), ('Missouri',), ('Musashi',), ('New Jersey',), ('North Carolina',), ('Ramillies',), ('Renown',), ('Repulse',), ('Resolution',), ('Revenge',), ('Royal Oak',), ('Royal Sovereign',), ('Tennessee',), ('Washington',), ('Wisconsin',), ('Yamato',)]


# Challenge

-- Find the country with the most powerful fleet (highest total displacement)

In [ ]:
cur.execute("""
    SELECT c.country, SUM(c.displacement) as total_displacement
        FROM Classes c
        JOIN Ships s ON c.class = s.class
        GROUP BY c.country
        ORDER BY total_displacement DESC
        LIMIT 1;
""")
results = cur.fetchall()
print(results)